In [18]:
import os
import logging
import argparse
import pandas as pd
from dotenv import load_dotenv
from neo4j_handler import Neo4jHandler

# Import Splink components using the confirmed syntax
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on, splink_datasets

# Configure logging.
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Load environment variables from the .env file.
load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
if not (NEO4J_URI and NEO4J_USER and NEO4J_PASSWORD):
    raise ValueError("Please set NEO4J_URI, NEO4J_USER, and NEO4J_PASSWORD in your .env file")

handler = Neo4jHandler(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

logger.info("Initializing DuckDB backend for Splink...")
db_api = DuckDBAPI()  # Create a DuckDB backend instance
from splink.exploratory import profile_columns

INFO:neo4j_handler:Connected to Neo4j at bolt://localhost:7687 as user neo4j
INFO:__main__:Initializing DuckDB backend for Splink...


In [19]:
def fetch_identities(handler):
    """
    Fetch all Identity nodes from the Neo4j database and return them as a list of dictionaries.
    Expected columns: id, full_name, email_address, zip_code, phone_number.
    """
    query = """
    MATCH (i:Identity)
    RETURN distinct i
    """
    result = handler.execute_read(query)
    # Consume the result within the transaction scope.
    records = [record.data() for record in result]
    return records


identities = fetch_identities(handler)

In [20]:
data = [identity["i"] for identity in identities]  
identities_df = pd.DataFrame(data)


identities_df.head()



,email_address,wcc_component,last_name,phone_number,id,full_address,first_name
0,moreyHolmes@gmail.com,0,Holmes,2806901865,895a9541-fa30-4d6d-836b-008ecff40135,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
1,moreyHolmes@gmail.com,0,Holmes,2806901865,642d0174-c5d5-4666-854c-1d9f790e32c4,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
2,moreyHolmes@gmail.com,0,Holmes,2806901865,297420aa-b8c1-4274-8f91-e5cf0619a421,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
3,moreyHolmes@gmail.com,0,Holmes,2806901865,e3c69cc4-62cc-4bea-be66-b3079e5d37f7,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
4,NaN,0,qolmes,2806901865,86b0cfd4-5a10-477e-ba48-6eaab01b21a0,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice


In [21]:
identities_df = identities_df.rename(columns={"id": "unique_id"})
identities_df.head()

,email_address,wcc_component,last_name,phone_number,unique_id,full_address,first_name
0,moreyHolmes@gmail.com,0,Holmes,2806901865,895a9541-fa30-4d6d-836b-008ecff40135,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
1,moreyHolmes@gmail.com,0,Holmes,2806901865,642d0174-c5d5-4666-854c-1d9f790e32c4,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
2,moreyHolmes@gmail.com,0,Holmes,2806901865,297420aa-b8c1-4274-8f91-e5cf0619a421,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
3,moreyHolmes@gmail.com,0,Holmes,2806901865,e3c69cc4-62cc-4bea-be66-b3079e5d37f7,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice
4,NaN,0,qolmes,2806901865,86b0cfd4-5a10-477e-ba48-6eaab01b21a0,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice


In [22]:

profile_columns(identities_df, db_api, column_expressions=["first_name", "last_name", "email_address", "phone_number",])

alt.VConcatChart(...)

In [23]:
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
)

blocking_rules = [
    block_on("wcc_component"),
    block_on("first_name"),
    block_on("last_name"),
    block_on("email_address"),
    block_on("phone_number")  
    
]


cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=identities_df,
    blocking_rules=blocking_rules,
    db_api=db_api,
    link_type="dedupe_only",
)

alt.Chart(...)

In [24]:


# Add a source_dataset column to the DataFrame.
identities_df["source_dataset"] = "dataset_1"  # Assign a default value for all records.

logger.info(f"Fetched {len(identities_df)} identity records.")

# Create Splink settings using the new syntax.
# Here we use our existing columns:
# - NameComparison on full_name
# - EmailComparison on email_address
# - ExactMatch on zip_code and phone_number (with term frequency adjustments for zip_code)
settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        # cl.NameComparison("full_name"),
        # cl.EmailComparison("email_address"),

        cl.ForenameSurnameComparison(
            "first_name",
            "last_name",
            forename_surname_concat_col_name="first_name_last_name_concat",
        ),
        cl.LevenshteinAtThresholds("email_address"),
        cl.ExactMatch("phone_number"),
        cl.LevenshteinAtThresholds("full_address")
    ],
    #     cl.LevenshteinAtThresholds("full_name").configure(term_frequency_adjustments=True),
    #     cl.LevenshteinAtThresholds("email_address").configure(term_frequency_adjustments=True),
    #     cl.LevenshteinAtThresholds("zip_code").configure(term_frequency_adjustments=True),
    #     cl.LevenshteinAtThresholds("phone_number").configure(term_frequency_adjustments=True),
    # ],
    retain_intermediate_calculation_columns=True,
    blocking_rules_to_generate_predictions=blocking_rules,
)


identities_df["first_name_last_name_concat"] = identities_df["first_name"] + " " + identities_df["last_name"]

logger.info("Initializing Splink Linker...")
# Initialize the generic Linker with the DataFrame, settings, and DuckDB backend.
linker = Linker(identities_df, settings, db_api=db_api)



INFO:__main__:Fetched 6352 identity records.
INFO:__main__:Initializing Splink Linker...


In [25]:
# -------------------------
# Training steps:
# Estimate the probability that two random records match.
linker.training.estimate_probability_two_random_records_match(
    [block_on("wcc_component")],
    recall=0.6,
)

INFO:splink.internals.linker_components.training:Probability two random records match is estimated to be  0.014.
This means that amongst all possible pairwise record comparisons, one in 71.44 are expected to match.  With 20,170,776 total possible comparisons, we expect a total of around 282,348.33 matching pairs


In [26]:
# Estimate u probabilities using random sampling.
linker.training.estimate_u_using_random_sampling(max_pairs=1e8)


INFO:splink.internals.estimate_u:----- Estimating u probabilities using random sampling -----
INFO:splink.internals.m_u_records_to_parameters:u probability not trained for first_name_last_name - Match on reversed cols: first_name and last_name (both directions) (comparison vector value: 5). This usually means the comparison level was never observed in the training data.
INFO:splink.internals.estimate_u:
Estimated u probabilities using random sampling
INFO:splink.internals.settings:
Your model is not yet fully trained. Missing estimates for:
    - first_name_last_name (some u values are not trained, no m values are trained).
    - email_address (no m values are trained).
    - phone_number (no m values are trained).
    - full_address (no m values are trained).


In [27]:

training_blocking_rule = block_on("wcc_component")
trainin_sesion_wcc= (
    linker.training.estimate_parameters_using_expectation_maximisation(
        training_blocking_rule, estimate_without_term_frequencies=True
    )
)

INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
l."wcc_component" = r."wcc_component"

Parameter estimates will be made for the following comparison(s):
    - first_name_last_name
    - email_address
    - phone_number
    - full_address

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
INFO:splink.internals.expectation_maximisation:
Level Match on reversed cols: first_name and last_name (both directions) on comparison first_name_last_name not observed in dataset, unable to train m value

INFO:splink.internals.expectation_maximisation:Iteration 1: Largest change in params was 0.158 in probability_two_random_records_match
INFO:splink.internals.expectation_maximisation:Iteration 2: Largest change in params was -0.0123 in the m_probability of email_address, level `Exact match on email_addres

In [28]:
linker.visualisations.match_weights_chart()

alt.VConcatChart(...)

In [29]:
linker.evaluation.unlinkables_chart()

alt.LayerChart(...)

In [ ]:
df_predict = linker.inference.predict()
df_e = df_predict.as_pandas_dataframe()
df_e = df_e[abs(df_e["match_probability"]) < .9]
df_e = df_e[abs(df_e["match_probability"]) > .5]

# df_e = df_e[df_e["match_weig"] > 0.1]

INFO:splink.internals.linker_components.inference:Blocking time: 0.28 seconds
INFO:splink.internals.linker_components.inference:Predict time: 5.77 seconds
 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'first_name_last_name':
    m values not fully trained
Comparison: 'first_name_last_name':
    u values not fully trained


match_weight                        1174
match_probability                   1174
unique_id_l                         1174
unique_id_r                         1174
last_name_l                         1174
last_name_r                         1174
first_name_l                        1174
first_name_r                        1174
first_name_last_name_concat_l       1174
first_name_last_name_concat_r       1174
gamma_first_name_last_name          1174
tf_first_name_last_name_concat_l    1174
tf_first_name_last_name_concat_r    1174
tf_last_name_l                      1174
tf_last_name_r                      1174
tf_first_name_l                     1174
tf_first_name_r                     1174
bf_first_name_last_name             1174
bf_tf_adj_first_name_last_name      1174
email_address_l                      599
email_address_r                      544
gamma_email_address                 1174
bf_email_address                    1174
phone_number_l                       545
phone_number_r  

In [ ]:
records_to_plot = df_e.sample(20).to_dict(orient="records")
linker.visualisations.waterfall_chart(records_to_plot, filter_nulls=False)

alt.LayerChart(...)

In [32]:
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=0.95
)

INFO:splink.internals.connected_components:Completed iteration 1, num representatives needing updating: 6
INFO:splink.internals.connected_components:Completed iteration 2, num representatives needing updating: 0


In [33]:
df_clusters = clusters.as_pandas_dataframe()
df_clusters.head()

,cluster_id,email_address,wcc_component,last_name,phone_number,unique_id,full_address,first_name,source_dataset,first_name_last_name_concat
0,0d587181-3e2d-48d7-ac72-6c4ef5b9daa1,moreyHolmes@gmail.com,0,Holmes,2806901865,895a9541-fa30-4d6d-836b-008ecff40135,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice,dataset_1,Maurice Holmes
1,0d587181-3e2d-48d7-ac72-6c4ef5b9daa1,moreyHolmes@gmail.com,0,Holmes,2806901865,642d0174-c5d5-4666-854c-1d9f790e32c4,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice,dataset_1,Maurice Holmes
2,0d587181-3e2d-48d7-ac72-6c4ef5b9daa1,moreyHolmes@gmail.com,0,Holmes,2806901865,297420aa-b8c1-4274-8f91-e5cf0619a421,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice,dataset_1,Maurice Holmes
3,0d587181-3e2d-48d7-ac72-6c4ef5b9daa1,moreyHolmes@gmail.com,0,Holmes,2806901865,e3c69cc4-62cc-4bea-be66-b3079e5d37f7,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice,dataset_1,Maurice Holmes
4,0d587181-3e2d-48d7-ac72-6c4ef5b9daa1,None,0,qolmes,2806901865,86b0cfd4-5a10-477e-ba48-6eaab01b21a0,"6592 Nowlin Rd, Seymour, New Hampshire 47181",Maurice,dataset_1,Maurice qolmes


In [34]:
# write the clusters to neo4j
rows = df_clusters.to_dict("records")

# Cypher query to update nodes: it unwinds each row and sets the cluster_id property.
update_query = """
UNWIND $rows as row
MATCH (n {id: row.id})
SET n.cluster_id = row.cluster_id
RETURN count(n) AS updated_count
"""

# Use your Neo4jHandler's execute_write method to run the update.
handler.execute_write(update_query, parameters={"rows": rows})